# Before You Start

This notebook runs the Civic Connect AI service temporarily in Google Colab for a live mobile demo.

## Required setup

1. Open **Runtime > Change runtime type** and select Python 3.
2. Use a CPU runtime or a T4 GPU runtime if one is available.
3. Run every code cell from top to bottom.
4. Wait until the final cell prints `classifier: ready` and an `AI_SERVICE_URL`.
5. Open the printed `/health` URL and confirm it returns HTTP 200.
6. In Render, open the `civic-connect-api` service and set `AI_SERVICE_URL` to the printed tunnel URL. Do not add `/health` or `/api`.
7. Redeploy the Node API after changing the environment variable.
8. Keep this Colab tab and runtime running while using the Flutter app outside your laptop.

The phone continues calling the Render API through `API_BASE_URL`. Render calls this temporary Colab service through Cloudflare Tunnel. The tunnel URL changes if Colab restarts, so update Render and redeploy the Node API whenever that happens.

This setup is for demonstrations only. Colab may disconnect, reclaim the runtime, or require the model to download again.

# Civic Connect AI Demo

This notebook starts the FastAPI vision service in Google Colab and exposes it through a temporary Cloudflare HTTPS tunnel.

Keep this tab open while demonstrating the mobile app. Copy the printed tunnel URL into the Render Node API environment variable `AI_SERVICE_URL`.

## Important demo limits

- Colab is temporary and may disconnect.
- The tunnel URL changes whenever the runtime restarts.
- The phone does not need to be on the same Wi-Fi network.
- Do not use this setup for production or sensitive data.

In [1]:
# Clone the repository into the Colab runtime.
# Change REPO_URL if the repository is private or has moved.
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Varunpatel586/Civic_Connect.git'
WORKSPACE = Path('/content/Civic_Connect')

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)

subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORKSPACE)], check=True)
AI_DIR = WORKSPACE / 'ai_service'
os.chdir(AI_DIR)
print(f'AI service directory: {AI_DIR}')

AI service directory: /content/Civic_Connect/ai_service


In [ ]:
# Install the AI service dependencies. This can take several minutes.
subprocess.run(['python', '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
print('Dependencies installed.')

In [ ]:
# Download cloudflared for the Colab Linux runtime.
CLOUDFLARED = Path('/content/cloudflared')
if not CLOUDFLARED.exists():
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', str(CLOUDFLARED),
    ], check=True)
    CLOUDFLARED.chmod(0o755)
print(f'cloudflared ready: {CLOUDFLARED}')

In [ ]:
# Start FastAPI and wait until CLIP has finished loading.
import time
import urllib.request

API_PROCESS = subprocess.Popen([
    'uvicorn', 'main:app',
    '--host', '0.0.0.0',
    '--port', '8000',
], cwd=AI_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

ready = False
deadline = time.time() + 600
while time.time() < deadline:
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=5) as response:
            health = response.read().decode()
            print(health)
            if 'ready' in health:
                ready = True
                break
    except Exception:
        pass
    time.sleep(5)

if not ready:
    API_PROCESS.terminate()
    raise RuntimeError('FastAPI did not become ready within 10 minutes.')

print('FastAPI and CLIP are ready.')

In [ ]:
# Create the temporary public HTTPS tunnel.
TUNNEL_LOG = Path('/content/cloudflared.log')
TUNNEL_PROCESS = subprocess.Popen([
    str(CLOUDFLARED), 'tunnel',
    '--url', 'http://127.0.0.1:8000',
    '--no-autoupdate',
    '--logfile', str(TUNNEL_LOG),
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

tunnel_url = None
deadline = time.time() + 60
while time.time() < deadline:
    if TUNNEL_LOG.exists():
        log = TUNNEL_LOG.read_text(errors='ignore')
        for token in log.split():
            if token.startswith('https://') and 'trycloudflare.com' in token:
                tunnel_url = token.rstrip('\"\,')
                break
    if tunnel_url:
        break
    time.sleep(2)

if not tunnel_url:
    raise RuntimeError('Cloudflare did not provide a tunnel URL. Check /content/cloudflared.log.')

print('AI_SERVICE_URL = ' + tunnel_url)
print('Health check: ' + tunnel_url + '/health')
print('Set this URL on the Render Node API, then redeploy the Node service.')

## Mobile demo checklist

1. Confirm the tunnel URL returns `classifier: ready` at `/health`.
2. In Render, set `AI_SERVICE_URL` to the printed URL without `/health`.
3. Redeploy the Node API.
4. Fully restart the Flutter app if its API configuration changed.
5. Take a pothole photograph from the phone.
6. Leave this Colab runtime running during the demo.

The Flutter app continues using the Render API URL. It should not call the Colab URL directly.